In [1]:
from app import Chatbot,PDFExtractor,DatabaseManager,DBTools,ChatBotUI,Config

e:\Projects\Agentic FinTracker\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
db=DatabaseManager(Config.DATABASE_PATH)
db_tools=DBTools(db)
chatbot=Chatbot(api_key=Config.API_KEY,model=Config.CHAT_BOT_MODEL,system_prompt=Config.CHAT_BOT_SYSTEM_PROMPT,tools_obj=db_tools)
pdf_extractor=PDFExtractor(api_key=Config.API_KEY,model=Config.PDF_EXTRACTOR_MODEL,system_prompt=Config.PDF_EXTRACTOR_SYSTEM_PROMPT)

calculate balance function


In [3]:
import gradio as gr
def chat(message, history, pdf_file=None):
    # If a PDF is uploaded, extract its content and prepend it to the message
    if pdf_file is not None:
        extracted = pdf_extractor.extract(pdf_file.name)
        if extracted:
            message = f"I have uploaded a PDF. Here is its extracted content:\n{extracted}\n\nUser message: {message}"
        else:
            message = f"{message}\n\n[Note: A PDF was uploaded but could not be parsed.]"
        return chatbot.send_message(message), gr.update(value=None, visible=False)

    return chatbot.send_message(message),gr.update()

In [ ]:
ui=ChatBotUI(chat_function=chat)
ui.lanuch_chat_UI()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


[{'role': 'system', 'content': "You are a personal financial assistant. Help users track their income and expenses using the available tools.\nYou can:\n- Log salary: use set_salary tool\n- Log expenses: use log_expense tool\n- Check balance: use get_balance tool\n- Get expense summary: use get_expense_summary tool\n\nWhen a user uploads a bank statement PDF, the extracted data will be provided to you in JSON format with salary and expenses. Automatically log each record one by one using the appropriate tools without asking for confirmation. After logging all records, give the user a summary of what was logged.\n\nWhen logging expenses use only these categories: Food, Transport, Utilities, Rent, Healthcare, Shopping, Entertainment, Education, Insurance, Subscriptions, Fuel, Other.\n\nAlways be concise and friendly. If the user asks anything unrelated to personal finance, politely let them know you can only help with financial tracking.\n\nToday's date is 2026-05-04."}, {'role': 'user',